# TinyLlama Roleplay Fine-Tuning (Refactored)

Breakdown:
1. Environment setup
2. Config
3. Training (single-stage SFT)
4. Prompt-system smoke test


## 1) Environment setup

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece faiss-cpu sentence-transformers PyYAML


In [ ]:
import sys
sys.path.append('.')


## 2) Config

In [ ]:
BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR = './tinyllama-roleplay-lora'
MAX_SEQ_LEN = 1024
MAX_SAMPLES_PER_DATASET = 30000
SEED = 42


## 3) Training (single-stage SFT)

Preserves the original notebook behavior:
- QLoRA 4-bit NF4
- LoRA targets attention+MLP projections
- Datasets: OASST1, ShareGPT Vicuna unfiltered, CharacterAI


In [ ]:
from training.single_stage_sft import run_single_stage_sft
from utils.logging import configure_logging

logger = configure_logging('INFO', 'roleplay_training')
run_single_stage_sft(
    base_model=BASE_MODEL,
    output_dir=OUTPUT_DIR,
    max_seq_len=MAX_SEQ_LEN,
    max_samples_per_dataset=MAX_SAMPLES_PER_DATASET,
    seed=SEED,
    trust_remote_code=True,
    logger=logger,
)


## 4) Prompt-system smoke test

Verifies Character Engine + World State + Emotion + Prompt Builder assembly.


In [ ]:
from characters.profile import load_character_profile_by_id
from emotion_engine.engine import EmotionState
from prompt_builder.builder import PromptBuilder
from world_state.state import WorldState

character_id = 'wizard'
character = load_character_profile_by_id(character_id)
world = WorldState(location='Arcane Forest', time='dusk', characters_present=[character.name], current_event='A strange fog rolls in', story_progress='in_progress')
emotions = EmotionState(by_character={character_id: 'curious'})

pb = PromptBuilder()
prompt = pb.build(
    character=character,
    world_state=world,
    emotion_state=emotions,
    character_id=character_id,
    retrieved_memories=['User previously asked about the forbidden library.'],
    chat_history=[('Where are we?', 'We stand beneath ancient boughs veiled in mist.')],
    user_input='What do you sense in this fog?'
)
print(prompt[:2000])
